# **RUNBOOK SIM2SIM V2**

Creado por Ph.D. Crisóstomo A. Barajas S. dentro del proyecto BLEA-ROBOTIKA, GIDSAW

Octubre, 2026

---

**Cambios respecto a v1:**
- §6 documenta explícitamente la bifurcación del pipeline (rama deploy vs rama entrenamiento)
- §7 es **puerta bloqueante** — sin §7 cerrado no se avanza a deploy físico, sin excepciones
- §7.2 incluye paso de copia explícita a la carpeta de referencia de deploy
- Nueva advertencia en §6: `motionlib_derived/` es un camino muerto para CSVs de deploy
- §9 (nuevo): export ONNX desde checkpoint, verificación de shapes, eval IsaacSim, prep TRT Jetson

---

# Sección 1 — Hotspot G1Capture

**Precondición:** Estás en 5061robotika como udirobotika. El robot está apagado todavía — el hotspot se levanta primero.

## 1.1 Verificar si la conexión ya existe

In [ ]:
nmcli con show G1Capture

Si devuelve datos → ir directo a 1.3. Si dice Error: ... not found → ejecutar 1.2.

## 1.2 Crear la conexión (solo primera vez)

In [ ]:
nmcli con add type wifi ifname wlp130s0f0 con-name G1Capture autoconnect no ssid G1Capture
nmcli con modify G1Capture \
  802-11-wireless.mode ap \
  802-11-wireless-security.key-mgmt wpa-psk \
  802-11-wireless-security.psk robotika2026 \
  ipv4.method shared \
  ipv4.addresses 10.42.0.1/24

## 1.3 Activar el hotspot

In [ ]:
nmcli con up G1Capture

## 1.4 Verificar que quedó bien

In [ ]:
ip addr show wlp130s0f0

Debe aparecer inet 10.42.0.1/24. Si no → revisar que no haya otro proceso usando la interfaz (nmcli dev status).

✅ Listo cuando: ip addr muestra 10.42.0.1/24 en wlp130s0f0.

---

# Sección 2 — Conectar Pico 4 al hotspot y verificar

**Precondición:** Hotspot G1Capture activo (10.42.0.1/24). Pico 4 encendida.

## 2.1 En la Pico 4 (manual, en el headset)

In [ ]:
Settings → Wi-Fi → G1Capture → contraseña: robotika2026

Esperar hasta que el ícono de WiFi quede estable.

## 2.2 Verificar desde la máquina que la Pico 4 tomó IP

In [ ]:
ip neigh show dev wlp130s0f0

Debe aparecer una entrada 10.42.0.x con estado REACHABLE. Anota esa IP — la necesitarás en la Sección 3.

En algunas ocasiones la IP de las Pico 4 queda como 10.42.0.67

## 2.3 Confirmar conectividad

In [ ]:
ping -c 3 10.42.0.67

✅ Listo cuando: ping responde limpio y tienes la IP de la Pico 4 anotada.

Sin pérdida de paquetes → listo. Si falla, revisar que la Pico 4 no se haya conectado a otra red disponible en el entorno.

## 2.4 Vestir trackers y calibrar en Pico 4

✅ Listo cuando: todos los trackers aparecen verdes en el headset y el movimiento del actor se refleja correctamente en la vista de Pico.

---

# Sección 3 — XRoboToolkit PC Service + .venv_teleop

**Precondición:** Hotspot activo, Pico 4 en 10.42.0.67, trackers calibrados y verdes, "Send" activo en XRoboToolkit app.

Dependencias Qt6 (solo primera vez)

In [ ]:
sudo apt install libqt6core5compat6 libqt6network6 libqt6widgets6 libqt6gui6 libqt6qml6 libqt6quick6 libqt6websockets6 libqt6serialport6

Verificar puertos limpios antes de arrancar

In [ ]:
ss -tulpn | grep -E "5556|5557|60061"

Si aparece algo → limpiar:

In [ ]:
fuser -k 5556/tcp 5557/tcp 60061/tcp

## 3.0 Terminal 0 — XRoboToolkit PC Service

In [ ]:
cd /opt/apps/roboticsservice
./RoboticsServiceProcess

**Dejar esta terminal corriendo — no cerrar. Abrir una terminal nueva para el resto de pasos.**

Debe mostrar release mode y quedarse bloqueado esperando. Si sale al prompt → algo falló.

## 3.1 Terminal 1 — MuJoCo sim loop

In [ ]:
cd ~/blea/GR00T-WholeBodyControl
source .venv_teleop/bin/activate
python gear_sonic/scripts/run_sim_loop.py

Esperar hasta que MuJoCo abra y el G1 virtual esté de pie. Los warnings de Wayland/OpenGL son normales.

## 3.2 Terminal 2 — Deploy ONNX

In [ ]:
cd ~/blea/GR00T-WholeBodyControl/gear_sonic_deploy
source scripts/setup_env.sh
./deploy.sh --input-type zmq_manager sim

Confirmar y cuando pregunte. Esperar "Init done" antes de continuar.

## 3.3. Terminal 3 — Pico manager

In [ ]:
cd ~/blea/GR00T-WholeBodyControl
source .venv_teleop/bin/activate
python gear_sonic/scripts/pico_manager_thread_server.py --manager

## 3.4 Activar captura en el headset (manual)

1. En XRoboToolkit app → habilitar "Send" checkbox (Data & Control)
2. Posición T-pose → presionar A+B+X+Y → esperar 1 segundo
3. Presionar A+X → iniciar stream
4. En T3 debe aparecer: StreamMode switch: PLANNER -> POSE

⚠️ Sin "Send" activo los botones no tienen efecto.

✅ Listo cuando: G1 virtual replica el movimiento del actor en tiempo real.

---

# Sección 4 — Captura y grabación

**Precondición:** Pipeline de 4 terminales activo, G1 virtual replicando movimiento del actor en tiempo real.

✅ La grabación ya está activa desde que T2 levantó. No hay que hacer nada extra para activarla.

# 4.1 Protocolo de captura

1. Actor en posición neutral → respirar, estabilizar
2. Realizar el movimiento completo 3–5 veces con pausas de 2–3 segundos entre repeticiones
3. Volver a posición neutral al final de cada repetición
4. Mantener ropa ajustada — ropa holgada afecta los trackers de tobillo

⚠️ Cada vez que se lanza T2 se crea una carpeta nueva con timestamp. Una sesión = una carpeta.

# 4.2 Terminar la sesión

T3 → Ctrl+C
T2 → Ctrl+C (o tecla O para emergency stop primero)
T1 → Ctrl+C
T0 → Ctrl+C

⚠️ Siempre en ese orden — T3 primero, T0 último.

# 4.3 Anotar el timestamp de la sesión

In [ ]:
ls ~/blea/GR00T-WholeBodyControl/gear_sonic_deploy/logs/

Anotar la carpeta DD-MM-YY/HH-MM-SS correspondiente a la sesión recién grabada. Se necesita en la Sección 5.

# 4.4 Verificar que q.csv tiene datos

In [ ]:
wc -l ~/blea/GR00T-WholeBodyControl/gear_sonic_deploy/logs/29-09-26/15-33-56/q.csv

Debe tener más de 100 líneas. Si tiene menos → la sesión fue muy corta o el stream no llegó.

✅ Listo cuando: q.csv tiene datos y el timestamp está anotado.

---

# Sección 5 — Verificación del replay

## 5.0 Definir la sesión a verificar

⚠️ MF: este es el único lugar donde debes escribir el timestamp. Copia la carpeta exacta del paso 4.3 y pégala aquí antes de correr cualquier comando de esta sección.

In [ ]:
# Formato: DD-MM-YY/HH-MM-SS
SESSION="29-09-26/15-12-55"

Verificar que existe:

In [ ]:
ls ~/blea/GR00T-WholeBodyControl/gear_sonic_deploy/logs/$SESSION/

A partir de aquí todos los comandos usan $SESSION — mientras la terminal esté abierta no hay que volver a definirla.

## 5.1 Revisar q.csv básico

In [ ]:
wc -l ~/blea/GR00T-WholeBodyControl/gear_sonic_deploy/logs/$SESSION/q.csv
head -10 ~/blea/GR00T-WholeBodyControl/gear_sonic_deploy/logs/$SESSION/q.csv

wc -l da el número de frames. Debe ser mayor a 100. head -3 confirma el formato — debe mostrar header con columnas index, time_ms, ..., q_0, ..., q_28.

✅ Listo cuando: más de 100 frames y header visible con 29 columnas q_.

## 5.2 Replay visual en MuJoCo

In [ ]:
cd ~/blea/GR00T-WholeBodyControl
source .venv_teleop/bin/activate
python - << 'EOF'
import mujoco
import mujoco.viewer
import pandas as pd
import time

SESSION="29-09-26/15-33-56"
BASE = f"/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/logs/{SESSION}"
XML  = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic/data/robots/g1/scene_29dof_freebase.xml"

# --- CONFIGURACIÓN DE RANGO DE FRAMES ---
START_FRAME = 1000        # Frame donde empieza la reproducción (ej: 100)
END_FRAME = None       # Frame donde termina (ej: 500, o None para ir hasta el final)
# ----------------------------------------

q = pd.read_csv(f"{BASE}/q.csv")
q_vals = q[[c for c in q.columns if c.startswith("q_")]].values

total_frames = len(q_vals)
print(total_frames)

# Ajustar los límites según corresponda
start = max(0, START_FRAME)
end = min(total_frames, END_FRAME) if END_FRAME is not None else total_frames
subset_vals = q_vals[start:end]

print(f"Reproduciendo del frame {start} al {end} (Total en rango: {len(subset_vals)} frames)...")

model = mujoco.MjModel.from_xml_path(XML)
data  = mujoco.MjData(model)

with mujoco.viewer.launch_passive(model, data) as viewer:
    # Usamos start=start para que el índice 'i' conserve su numeración real respecto al CSV original
    for i, frame in enumerate(subset_vals, start=start):
        # Muestra el número de frame cada 50 iteraciones
        if i % 50 == 0:
            print(f"Reproduciendo frame: {i} / {total_frames}")

        data.qpos[7:] = frame
        mujoco.mj_forward(model, data)
        viewer.sync()
        time.sleep(0.02)

print("¡Reproducción finalizada!")
EOF

⚠️ El replay muestra el G1 con root fijo en el origen — los pies pueden flotar ligeramente. Esto es normal: q.csv no contiene la traslación del root. Lo relevante es la forma del movimiento articular, no la posición global.

## 5.3 Graficar norma de velocidades angulares

In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

SESSION="29-09-26/15-33-56"
BASE = f"/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/logs/{SESSION}"

dq = pd.read_csv(f"{BASE}/dq.csv")
dq_vals = dq[[c for c in dq.columns if c.startswith("dq_")]].values
norm = np.linalg.norm(dq_vals, axis=1)

fig = go.Figure()
fig.add_trace(go.Scatter(y=norm, mode='lines', name='|dq| total'))
fig.add_hline(y=20, line_dash="dash", line_color="red", annotation_text="threshold=20")

fig.update_layout(
    title="Norma de velocidades angulares por frame",
    xaxis_title="Frame",
    yaxis_title="|dq| total",
    template="plotly_white"
)
fig.show()

Abrir dq_norm.png e identificar visualmente el frame donde termina el ruido inicial y la señal se estabiliza bajo el threshold rojo.

## 5.4 Definir segmento limpio

⚠️ MF: editar FRAME_START con el valor que identificaste en el gráfico.

In [ ]:
FRAME_START=1000   # <- reemplazar con el frame leído del gráfico
FRAME_END=5700    # <- reemplazar con el frame leído del gráfico

## 5.5 Re-verificar alertas sobre segmento limpio

In [ ]:
python - << 'EOF'
import pandas as pd
import numpy as np

SESSION="29-09-26/15-33-56"
FRAME_START=820  # <- reemplazar con el frame leído del gráfico
BASE = f"/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/logs/{SESSION}"

dq = pd.read_csv(f"{BASE}/dq.csv")
dq_vals = dq[[c for c in dq.columns if c.startswith("dq_")]].values
dq_clean = dq_vals[FRAME_START:]

THRESHOLD = 10.0
print(f"Frames analizados: {len(dq_clean)} (desde frame {FRAME_START})")
print(f"\n{'Joint':<10} {'Max |dq|':>10} {'Alerta':>8}")
print("-" * 30)
problemas = []
for i in range(29):
    peak = np.abs(dq_clean[:, i]).max()
    alerta = "⚠️" if peak > THRESHOLD else "✅"
    if peak > THRESHOLD:
        problemas.append((i, peak))
    print(f"joint_{i:<4} {peak:>10.3f} {alerta:>8}")

print()
if problemas:
    print("⚠️  Joints fuera de rango — revisar movimiento antes de continuar:")
    for joint, val in problemas:
        print(f"   joint_{joint}: {val:.2f} rad/s")
else:
    print("✅ Segmento limpio — listo para conversión PKL")
EOF

Si aparecen alertas en joints de brazos o torso → revisar replay visual del segmento. Si son solo piernas durante sentadilla profunda → aceptable.

---

# Bifurcación del pipeline en §6

A partir de §6 el pipeline se divide en **dos ramas independientes**:

§5 (sesión verificada)
│
▼
§6.1 → §6.2 — deploy PKL ──→ §7 (vis PKL + CSVs + gate visual) ──→ Deploy físico
§6.3 → §6.4 — motionlib PKL ──→ §8 (finetune Isaac Lab)


**Las dos ramas usan archivos distintos con formatos distintos.** No son intercambiables.

| | Deploy PKL (`robot_filtered/`) | Motionlib PKL (`robot_filtered_motionlib_*/`) |
|---|---|---|
| Keys | `joint_pos`, `body_pos_w`, `body_quat_w`, `joint_order='mj'` | `dof`, `root_trans_offset`, `root_rot`, `pose_aa`, `fps` |
| Quién lo usa | §7.1 → vis PKL → CSVs → C++ binary | §8 → Isaac Lab training |
| Joint order | MJ (tal como sale de q.csv) | MJ (convertido por `convert_soma_csv_to_motion_lib.py`) |

⚠️ **`motionlib_derived/`** contiene PKLs atómicos intermedios generados internamente por el training pipeline de Isaac Lab. **Nunca usar como fuente de CSVs de deploy** — no tienen la permutación IL↔MJ y producen arm "parálisis cerebral" en el robot.

---

# 6 — CONFIG de conversión PKL

Este es el único bloque que se edita en §6. Copia SESSION y FRAME_START/END
del §5.0 y §5.4 exactamente.

OUTPUT_NAME describe el movimiento de esta sesión. Es necesario verificar el nombre de esta variable en cada grabacion

In [ ]:
SESSION="29-09-26/15-33-56"   # copiar de §5.0
FRAME_START=820              # <- reemplazar con el frame leído del gráfico
FRAME_END=10330                # <- reemplazar con el frame leído del gráfico

OUTPUT_NAME="macarena2_cris_sostenido"  # movimiento_variante_actor (nombre largo con actor)
MOTION_KEY="macarena2"                     # clave dentro del PKL dict; también subdir de deploy
MOTION_NAME="macarena2"                    # carpeta base de referencia de deploy (sin actor/variante)
                                              # en la mayoría de los casos MOTION_NAME == MOTION_KEY
ROOT_HEIGHT=0.78
FPS=50

BASE_BLEA="/home/udirobotika/blea/GR00T-WholeBodyControl"
LOG_SESSION="${BASE_BLEA}/gear_sonic_deploy/logs/${SESSION}"
OUT_DEPLOY="${BASE_BLEA}/robot_filtered/${OUTPUT_NAME}.pkl"
OUT_MOTIONLIB="${BASE_BLEA}/robot_filtered_motionlib_clean/${OUTPUT_NAME}.pkl"
CONVERSOR="${BASE_BLEA}/gear_sonic/data_process/convert_soma_csv_to_motion_lib.py"

echo "Sesión   : ${LOG_SESSION}"
echo "Deploy   : ${OUT_DEPLOY}"
echo "Motionlib: ${OUT_MOTIONLIB}"
echo "Segmento : frames ${FRAME_START}–${FRAME_END}  $(( FRAME_END - FRAME_START )) frames"
echo ""
echo "CSVs de deploy irán a:"
echo "  ${BASE_BLEA}/gear_sonic_deploy/reference/${MOTION_NAME}/${MOTION_KEY}/"

✅ Listo cuando: los paths impresos existen y el conteo de frames es mayor a 200.

## 6.1 — Construcción del deploy PKL

In [ ]:
python - << 'EOF'
import pandas as pd, numpy as np, joblib, pathlib

SESSION="29-09-26/15-33-56"   # copiar de §5.0
FRAME_START=820              # <- reemplazar con el frame leído del gráfico
FRAME_END=10330                # <- reemplazar con el frame leído del gráfico
OUTPUT_NAME="macarena2_cris_sostenido"  # movimiento_variante_actor (nombre largo con actor)
MOTION_KEY="macarena2"                     # clave dentro del PKL dict; también subdir de deploy
ROOT_HEIGHT  = 0.78
FPS          = 50

BASE_BLEA    = "/home/udirobotika/blea/GR00T-WholeBodyControl"
LOG_SESSION  = f"{BASE_BLEA}/gear_sonic_deploy/logs/{SESSION}"
OUT_DEPLOY   = f"{BASE_BLEA}/robot_filtered/{OUTPUT_NAME}.pkl"

q_df  = pd.read_csv(f"{LOG_SESSION}/q.csv")
bq_df = pd.read_csv(f"{LOG_SESSION}/base_quat.csv")

# Alinear por 'index', no por posición de fila (seguro ante drops del ring buffer)
m   = (q_df['index'] >= FRAME_START) & (q_df['index'] <= FRAME_END)
idx = q_df.loc[m, 'index'].values

joint_pos = q_df.loc[m, [f'q_{i}' for i in range(29)]].values.astype(np.float32)
root_wxyz = bq_df.set_index('index').loc[idx,
              ['base_qw','base_qx','base_qy','base_qz']].values.astype(np.float32)

T = len(joint_pos)
assert len(root_wxyz) == T, f"desalineación: q={T} vs quat={len(root_wxyz)}"

root_wxyz /= np.linalg.norm(root_wxyz, axis=1, keepdims=True)

body_pos_w = np.zeros((T, 14, 3), dtype=np.float32)
body_pos_w[:, 0, 2] = ROOT_HEIGHT

body_quat_w = np.zeros((T, 14, 4), dtype=np.float32)
body_quat_w[:, :, 0] = 1.0
body_quat_w[:, 0, :] = root_wxyz

deploy_dict = {
    MOTION_KEY: {
        "joint_pos":   joint_pos,
        "body_pos_w":  body_pos_w,
        "body_quat_w": body_quat_w,
        "joint_order": "mj",   # q_0..q_28 ya están en orden MuJoCo — NO reordenar
    }
}

pathlib.Path(OUT_DEPLOY).parent.mkdir(parents=True, exist_ok=True)
joblib.dump(deploy_dict, OUT_DEPLOY, compress=True)

print(f"OK → {OUT_DEPLOY}")
print(f"T={T} frames  ({T/FPS:.1f}s @ {FPS}fps)")
print(f"joint_pos  : {joint_pos.shape}  {joint_pos.dtype}")
print(f"body_quat_w: {body_quat_w.shape}  {body_quat_w.dtype}")
print(f"root_wxyz[0]: {np.round(root_wxyz[0], 4)}  (wxyz, w en idx 0 debe ser ~0.99)")
EOF

✅ Listo cuando: imprime OK → con el path correcto y w ≈ 0.99.

## 6.2 — Verificación del deploy PKL:

In [ ]:
python - << 'EOF'
import joblib, numpy as np

OUTPUT_NAME="macarena2_cris_sostenido"  # movimiento_variante_actor (nombre largo con actor)
MOTION_KEY="macarena2"                     # clave dentro del PKL dict; también subdir de deploy
BASE_BLEA   = "/home/udirobotika/blea/GR00T-WholeBodyControl"
OUT_DEPLOY  = f"{BASE_BLEA}/robot_filtered/{OUTPUT_NAME}.pkl"

d   = joblib.load(OUT_DEPLOY)
seq = d[MOTION_KEY]

print("=== ESTRUCTURA ===")
print(f"keys PKL   : {list(d.keys())}")
print(f"campos     : {list(seq.keys())}")
print(f"joint_order: {seq.get('joint_order', 'NO PRESENTE')}  <- debe ser 'mj'")

print("\n=== SHAPES ===")
for k in ["joint_pos", "body_pos_w", "body_quat_w"]:
    v = np.asarray(seq[k])
    print(f"  {k}: {v.shape}  {v.dtype}")

print("\n=== QUATS ROOT ===")
rq    = np.asarray(seq["body_quat_w"])[:, 0, :]
norms = np.linalg.norm(rq, axis=1)
print(f"  norma: min={norms.min():.5f}  max={norms.max():.5f}  <- debe ser ~1.0")
print(f"  rq[0] = {np.round(rq[0], 4)}  (wxyz, w en idx 0 debe ser ~0.99)")

print("\n=== VARIANZA POR GRUPO ===")
jp = np.asarray(seq["joint_pos"])
print(f"  right arm (22-28): {np.round(jp[:, 22:29].std(0), 3)}")
print(f"  left arm  (15-21): {np.round(jp[:, 15:22].std(0), 3)}")
print(f"  legs      ( 0-11): {np.round(jp[:,  0:12].std(0), 3)}")
print(f"  waist     (12-14): {np.round(jp[:, 12:15].std(0), 3)}")
EOF

✅ Listo cuando: joint_order='mj', normas ~1.0.

## 6.3 — conversor oficial:

In [ ]:
conda activate isaaclab
python $CONVERSOR \
  --input  $OUT_DEPLOY \
  --output $OUT_MOTIONLIB \
  --fps    $FPS

✅ Listo cuando: imprime Done: 1 sequences saved sin errores.

## 6.4 — Verificación del motionlib PKL:

In [ ]:
python - << 'EOF'
import joblib, numpy as np

OUTPUT_NAME="macarena2_cris_sostenido"  # movimiento_variante_actor (nombre largo con actor)
MOTION_KEY="macarena2"                     # clave dentro del PKL dict; también subdir de deploy
BASE_BLEA   = "/home/udirobotika/blea/GR00T-WholeBodyControl"
OUT_MOTIONLIB = f"{BASE_BLEA}/robot_filtered_motionlib_clean/{OUTPUT_NAME}.pkl"

d   = joblib.load(OUT_MOTIONLIB)
seq = d[MOTION_KEY]

print("=== ESTRUCTURA ===")
print(f"campos: {list(seq.keys())}")
for k in ["root_trans_offset", "pose_aa", "dof", "root_rot", "smpl_joints"]:
    v = np.asarray(seq[k])
    print(f"  {k}: {v.shape}  {v.dtype}")
print(f"  fps: {seq['fps']}")

print("\n=== ROOT_ROT (xyzw) ===")
rr = np.asarray(seq["root_rot"])
norms = np.linalg.norm(rr, axis=1)
print(f"  norma: min={norms.min():.5f}  max={norms.max():.5f}  <- debe ser ~1.0")
print(f"  rr[0] = {np.round(rr[0], 4)}  (xyzw, w en idx 3 debe ser ~0.99)")

print("\n=== VARIANZA DOF POR GRUPO ===")
dof = np.asarray(seq["dof"])
print(f"  right arm (22-28): {np.round(dof[:, 22:29].std(0), 3)}")
print(f"  left arm  (15-21): {np.round(dof[:, 15:22].std(0), 3)}")
print(f"  legs      ( 0-11): {np.round(dof[:,  0:12].std(0), 3)}")
print(f"  waist     (12-14): {np.round(dof[:, 12:15].std(0), 3)}")

print("\n=== POSE_AA ===")
pa = np.asarray(seq["pose_aa"])
print(f"  shape: {pa.shape}  <- debe ser (T, 30, 3)")
EOF

✅ Listo cuando: todos los checks pasan. El PKL está listo para entrenamiento en Isaac Lab.

⛔ **ADVERTENCIA — `motionlib_derived/` es un dead end para CSVs de deploy**

Los PKLs en `motionlib_derived/` son fragmentos atómicos generados internamente por Isaac Lab durante el entrenamiento. Tienen un formato diferente al deploy PKL de §6.1 y **no tienen la permutación IL↔MJ** que el C++ binary espera.

Usar cualquier PKL de `motionlib_derived/` como fuente de CSVs de deploy produce movimiento de brazos espástico ("parálisis cerebral") en el robot físico — como se observó en la primera ejecución real el 2026-10-02.

**Ruta correcta:**

§6.1 deploy PKL → §7.1 vis PKL (aplica MJ→IL) → §7.2 convert_motions.py → CSVs correctos


Si §7.3 checklist no pasa con inspección visual → volver a §6.1, nunca a `motionlib_derived/`.

---

# Sección 7 — Gate visual del movimiento

**Precondición:** deploy PKL generado y verificado en §6.2. Display disponible.

## ⚠️ PUERTA BLOQUEANTE — Sin §7 cerrado no se avanza a deploy físico, sin excepciones

El gate de §7 existe para detectar el bug de "parálisis cerebral" **antes** de tocar el robot. El visualizador aplica internamente IL→MJ sobre los joints. Si el PKL visual tiene el orden incorrecto, el movimiento sale revuelto en MuJoCo — señal inequívoca de que los CSVs de deploy también estarán mal.

**Checklist de apertura de §7:**
- [ ] §6.1 completado — deploy PKL en `robot_filtered/`
- [ ] §6.2 verificado — `joint_order='mj'`, normas ~1.0, varianza de brazos no nula
- [ ] Display disponible para inspección visual (no correr en SSH puro sin -X)

Esta sección construye el PKL visual con la doble inversión MJ→IL, genera los CSVs de deploy, los copia a la carpeta de referencia, y cierra con inspección visual obligatoria.

## 7.1 — Construcción del PKL visual:

In [ ]:
export OUTPUT_NAME="macarena2_cris_sostenido" # <- editar por sesión: movimiento_variante_actor
export MOTION_KEY="macarena2"

export ROOT_HEIGHT="0.78"
export FPS="50"

export BASE_BLEA="/home/udirobotika/blea/GR00T-WholeBodyControl"
python - << 'EOF'
import joblib, numpy as np, pathlib

DIR_TRAINING="/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2" # <- carpeta con PKLs aprobados del skill
FPS_ESPERADO = 50

pkls = sorted(pathlib.Path(DIR_TRAINING).glob("*.pkl"))
print(f"Carpeta : {DIR_TRAINING}")
print(f"PKLs    : {len(pkls)}\n")

ok_count = 0
for pkl in pkls:
    try:
        d   = joblib.load(pkl)
        key = list(d.keys())[0]
        seq = d[key]
        dof = np.asarray(seq["dof"])
        rr  = np.asarray(seq["root_rot"])
        norms = np.linalg.norm(rr, axis=1)

        shape_ok = dof.shape[1] == 29
        quat_ok  = norms.min() > 0.999 and norms.max() < 1.001
        fps_ok   = seq.get("fps") == FPS_ESPERADO

        status = "✅" if (shape_ok and quat_ok and fps_ok) else "⚠️"
        if shape_ok and quat_ok and fps_ok:
            ok_count += 1
        print(f"{status} {pkl.name}")
        print(f"   key={key}  T={dof.shape[0]}  ({dof.shape[0]/FPS_ESPERADO:.1f}s @ {seq.get('fps')}fps)")
        print(f"   dof={dof.shape}  root_rot norm OK={quat_ok}")
        if not (shape_ok and quat_ok and fps_ok):
            print(f"   ⚠️ PROBLEMA — no incluir en entrenamiento")
    except Exception as e:
        print(f"⚠️ {pkl.name}  ERROR: {e}")

print(f"\n{'='*50}")
print(f"Resultado: {ok_count}/{len(pkls)} PKLs listos")
if ok_count == len(pkls) and len(pkls) > 0:
    print("→ Set completo. Proceder a §8.2.")
else:
    print("→ Revisar PKLs con ⚠️ antes de entrenar.")
EOF
export OUT_DEPLOY="${BASE_BLEA}/robot_filtered/${OUTPUT_NAME}.pkl"

export OUT_VIS="${BASE_BLEA}/gear_sonic_deploy/reference/${OUTPUT_NAME}_vis.pkl"

In [ ]:
python - << 'EOF'
import joblib, numpy as np, pathlib
import os

OUTPUT_NAME = "macarena2_cris_sostenido"   # <- editar por sesión: movimiento_variante_actor
MOTION_KEY  = "macarena2"
ROOT_HEIGHT = 0.78
FPS         = 50
BASE_BLEA   = "/home/udirobotika/blea/GR00T-WholeBodyControl"
OUT_DEPLOY  = f"{BASE_BLEA}/robot_filtered/{OUTPUT_NAME}.pkl"
OUT_VIS     = f"{BASE_BLEA}/gear_sonic_deploy/reference/{OUTPUT_NAME}_vis.pkl"

OUTPUT_NAME = os.environ["OUTPUT_NAME"]
MOTION_KEY  = os.environ["MOTION_KEY"]
ROOT_HEIGHT = float(os.environ["ROOT_HEIGHT"])
FPS         = int(os.environ["FPS"])
BASE_BLEA  = os.environ["BASE_BLEA"]
OUT_DEPLOY = os.environ["OUT_DEPLOY"]
OUT_VIS    = os.environ["OUT_VIS"]

# Permutaciones IL ↔ MJ
# El visualizador aplica IL→MJ internamente, así que pre-invertimos MJ→IL
IL_TO_MJ = np.array([0,3,6,9,13,17,1,4,7,10,14,18,2,5,8,
                      11,15,19,21,23,25,27,12,16,20,22,24,26,28], dtype=np.int32)
MJ_TO_IL = np.argsort(IL_TO_MJ)

seq     = joblib.load(OUT_DEPLOY)[MOTION_KEY]
dof_mj  = np.asarray(seq["joint_pos"],   dtype=np.float32)  # (T,29) MJ
rq_wxyz = np.asarray(seq["body_quat_w"], dtype=np.float32)  # (T,14,4) wxyz
T       = dof_mj.shape[0]

# Joints: MJ → IL (doble inversión: visualizador los regresa a MJ internamente)
dof_il = dof_mj[:, MJ_TO_IL]

# Root quat: wxyz → xyzw (visualizador espera xyzw)
rq_xyzw = rq_wxyz[:, 0, np.array([1,2,3,0])]  # (T,4) xyzw

body_pos_w = np.zeros((T, 14, 3), dtype=np.float32)
body_pos_w[:, 0, 2] = ROOT_HEIGHT

body_quat_w = np.zeros((T, 14, 4), dtype=np.float32)
body_quat_w[:, :, 3] = 1.0       # identidad xyzw (w en idx 3)
body_quat_w[:, 0, :] = rq_xyzw   # root real xyzw

# Velocidades: ceros — el visualizador exige que existan pero no las usa
joint_vel      = np.zeros((T, 29), dtype=np.float32)
body_lin_vel_w = np.zeros((T, 14, 3), dtype=np.float32)
body_ang_vel_w = np.zeros((T, 14, 3), dtype=np.float32)

vis_dict = {
    MOTION_KEY: {
        "joint_pos":      dof_il,# Verificación — comparar frames entre fuente y destino

export MOTION_NAME
export OUTPUT_NAME
export MOTION_KEY

python - << 'EOF'
import os, pathlib

OUTPUT_NAME = os.environ["OUTPUT_NAME"]
MOTION_NAME = os.environ["MOTION_NAME"]
MOTION_KEY  = os.environ["MOTION_KEY"]
BASE        = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy"

print(" ")
print(f"OUTPUT_NAME: {OUTPUT_NAME}")
print(f"MOTION_NAME: {MOTION_NAME}")
print(f"MOTION_KEY: {MOTION_KEY}")
print(" ")
print(" ")
src  = pathlib.Path(f"{BASE}/reference/{OUTPUT_NAME}_vis/{MOTION_KEY}")
dst  = pathlib.Path(f"{BASE}/reference/{MOTION_NAME}/{MOTION_KEY}")

src_files = sorted(src.glob("*.csv"))
dst_files = sorted(dst.glob("*.csv"))

print(f"Fuente  : {src}")
print(f"Destino : {dst}")
print()

if not dst.exists():
    print("⛔ Destino no existe — ejecutar paso 1b")
    raise SystemExit(1)

for f in src_files:
    d = dst / f.name# Verificación — comparar frames entre fuente y destino

export MOTION_NAME
export OUTPUT_NAME
export MOTION_KEY

python - << 'EOF'
import os, pathlib

OUTPUT_NAME = os.environ["OUTPUT_NAME"]
MOTION_NAME = os.environ["MOTION_NAME"]
MOTION_KEY  = os.environ["MOTION_KEY"]
BASE        = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy"

print(" ")
print(f"OUTPUT_NAME: {OUTPUT_NAME}")
print(f"MOTION_NAME: {MOTION_NAME}")
print(f"MOTION_KEY: {MOTION_KEY}")
print(" ")
print(" ")
src  = pathlib.Path(f"{BASE}/reference/{OUTPUT_NAME}_vis/{MOTION_KEY}")
dst  = pathlib.Path(f"{BASE}/reference/{MOTION_NAME}/{MOTION_KEY}")

src_files = sorted(src.glob("*.csv"))
dst_files = sorted(dst.glob("*.csv"))

print(f"Fuente  : {src}")
print(f"Destino : {dst}")
print()

if not dst.exists():
    print("⛔ Destino no existe — ejecutar paso 1b")
    raise SystemExit(1)

for f in src_files:
    d = dst / f.name
    src_lines = f.read_text().count('\n')
    dst_lines = d.read_text().count('\n') if d.exists() else -1
    match = "✅" if src_lines == dst_lines else "⛔"
    print(f"  {match}  {f.name}  ({src_lines} líneas)")

if len(src_files) == len(dst_files):
    print(f"\n✅ {len(dst_files)} archivos copiados correctamente")
    print("→ Listo para §7.2 Paso 2 (visualizador)")
else:
    print(f"\n⛔ Mismatch: fuente={len(src_files)} destino={len(dst_files)}")
EOF
    src_lines = f.read_text().count('\n')
    dst_lines = d.read_text().count('\n') if d.exists() else -1
    match = "✅" if src_lines == dst_lines else "⛔"
    print(f"  {match}  {f.name}  ({src_lines} líneas)")

if len(src_files) == len(dst_files):
    print(f"\n✅ {len(dst_files)} archivos copiados correctamente")
    print("→ Listo para §7.2 Paso 2 (visualizador)")
else:
    print(f"\n⛔ Mismatch: fuente={len(src_files)} destino={len(dst_files)}")
EOF
        "joint_vel":      joint_vel,
        "body_pos_w":     body_pos_w,
        "body_quat_w":    body_quat_w,
        "body_lin_vel_w": body_lin_vel_w,
        "body_ang_vel_w": body_ang_vel_w,
    }
}

pathlib.Path(OUT_VIS).parent.mkdir(parents=True, exist_ok=True)
joblib.dump(vis_dict, OUT_VIS, compress=True)

print(f"OK → {OUT_VIS}")
print(f"T={T}  ({T/FPS:.1f}s @ {FPS}fps)")
print(f"joint_pos (IL order) : {dof_il.shape}")
print(f"body_quat_w (xyzw)   : {body_quat_w.shape}")
print(f"root xyzw[0]         : {np.round(rq_xyzw[0], 4)}  (w en idx 3 debe ser ~0.99)")
print(f"round-trip MJ→IL→MJ  : {np.array_equal(dof_il[:, IL_TO_MJ], dof_mj)}  <- debe ser True")
EOF

✅ Listo cuando: imprime OK →, w ≈ 0.99, y round-trip True.

## 7.2 — Lanzar el visualizador:

Paso 1 — PKL visual → CSVs:

In [ ]:
# Paso 1 — PKL visual → CSVs
cd ~/blea/GR00T-WholeBodyControl
conda activate gear_sonic_teleop

python gear_sonic_deploy/reference/convert_motions.py \
  gear_sonic_deploy/reference/${OUTPUT_NAME}_vis.pkl

echo "CSVs generados en:"
echo "  gear_sonic_deploy/reference/${OUTPUT_NAME}_vis/${MOTION_KEY}/"
ls gear_sonic_deploy/reference/${OUTPUT_NAME}_vis/${MOTION_KEY}/

Paso 1b — Copia a carpeta de deploy

In [ ]:
# Paso 1b — Copiar CSVs a la carpeta de referencia de deploy
# ⚠️ Este paso es OBLIGATORIO. Sin él los CSVs correctos no llegan al binary.
# Fuente:  reference/${OUTPUT_NAME}_vis/${MOTION_KEY}/
# Destino: reference/${MOTION_NAME}/${MOTION_KEY}/

DEPLOY_REF="gear_sonic_deploy/reference/${MOTION_NAME}/${MOTION_KEY}"

mkdir -p "gear_sonic_deploy/reference/${MOTION_NAME}/"
rm -rf "${DEPLOY_REF}"
cp -r "gear_sonic_deploy/reference/${OUTPUT_NAME}_vis/${MOTION_KEY}" \
      "${DEPLOY_REF}"

echo "✅ Copia completada → ${DEPLOY_REF}/"
echo ""
echo "Verificando:"
ls "${DEPLOY_REF}/"
echo ""
cat "${DEPLOY_REF}/metadata.txt"

Paso 2 — Visualizar:

In [ ]:
cd ~/blea/GR00T-WholeBodyControl/gear_sonic_deploy
python visualize_motion.py \
  --motion_dir reference/${OUTPUT_NAME}_vis/${MOTION_KEY}/

⚠️ Requiere display activo. Si estás en SSH → agregar -X al conectar o correr directamente en la terminal del escritorio.

✅ Listo cuando: abre ventana con el G1 ejecutando el movimiento capturado.

Verificación de la copia

In [ ]:
# Verificación — comparar frames entre fuente y destino

export MOTION_NAME
export OUTPUT_NAME
export MOTION_KEY

python - << 'EOF'
import os, pathlib

OUTPUT_NAME = os.environ["OUTPUT_NAME"]# Verificación — comparar frames entre fuente y destino

export MOTION_NAME
export OUTPUT_NAME
export MOTION_KEY

python - << 'EOF'
import os, pathlib

OUTPUT_NAME = os.environ["OUTPUT_NAME"]
MOTION_NAME = os.environ["MOTION_NAME"]
MOTION_KEY  = os.environ["MOTION_KEY"]
BASE        = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy"

print(" ")
print(f"OUTPUT_NAME: {OUTPUT_NAME}")
print(f"MOTION_NAME: {MOTION_NAME}")
print(f"MOTION_KEY: {MOTION_KEY}")
print(" ")
print(" ")
src  = pathlib.Path(f"{BASE}/reference/{OUTPUT_NAME}_vis/{MOTION_KEY}")
dst  = pathlib.Path(f"{BASE}/reference/{MOTION_NAME}/{MOTION_KEY}")

src_files = sorted(src.glob("*.csv"))
dst_files = sorted(dst.glob("*.csv"))

print(f"Fuente  : {src}")
print(f"Destino : {dst}")
print()

if not dst.exists():
    print("⛔ Destino no existe — ejecutar paso 1b")
    raise SystemExit(1)

for f in src_files:
    d = dst / f.name
    src_lines = f.read_text().count('\n')
    dst_lines = d.read_text().count('\n') if d.exists() else -1
    match = "✅" if src_lines == dst_lines else "⛔"
    print(f"  {match}  {f.name}  ({src_lines} líneas)")

if len(src_files) == len(dst_files):
    print(f"\n✅ {len(dst_files)} archivos copiados correctamente")
    print("→ Listo para §7.2 Paso 2 (visualizador)")
else:
    print(f"\n⛔ Mismatch: fuente={len(src_files)} destino={len(dst_files)}")
EOF
MOTION_NAME = os.environ["MOTION_NAME"]
MOTION_KEY  = os.environ["MOTION_KEY"]
BASE        = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy"

print(" ")
print(f"OUTPUT_NAME: {OUTPUT_NAME}")
print(f"MOTION_NAME: {MOTION_NAME}")
print(f"MOTION_KEY: {MOTION_KEY}")
print(" ")
print(" ")
src  = pathlib.Path(f"{BASE}/reference/{OUTPUT_NAME}_vis/{MOTION_KEY}")
dst  = pathlib.Path(f"{BASE}/reference/{MOTION_NAME}/{MOTION_KEY}")

src_files = sorted(src.glob("*.csv"))
dst_files = sorted(dst.glob("*.csv"))

print(f"Fuente  : {src}")
print(f"Destino : {dst}")
print()

if not dst.exists():
    print("⛔ Destino no existe — ejecutar paso 1b")
    raise SystemExit(1)

for f in src_files:
    d = dst / f.name
    src_lines = f.read_text().count('\n')
    dst_lines = d.read_text().count('\n') if d.exists() else -1
    match = "✅" if src_lines == dst_lines else "⛔"
    print(f"  {match}  {f.name}  ({src_lines} líneas)")

if len(src_files) == len(dst_files):
    print(f"\n✅ {len(dst_files)} archivos copiados correctamente")
    print("→ Listo para §7.2 Paso 2 (visualizador)")
else:
    print(f"\n⛔ Mismatch: fuente={len(src_files)} destino={len(dst_files)}")
EOF

## 7.3 — Checklist visual

- [ ] Robot arranca vertical, no inclinado ni acostado
- [ ] Movimiento reconocible — se parece a lo que hizo el actor
- [ ] Sin teleportaciones bruscas entre frames
- [ ] Pies razonablemente estables (algo de flotación es normal — root fijo)

⚠️ Si el robot sale inclinado ~45° → problema de convención de frame del IMU. Reportar a Cris antes de continuar.

✅ Gate cerrado cuando: todos los ítems pasan con inspección visual. Recién ahí se mueven los PKLs aprobados a DIR_TRAINING y se procede a §8.

---

# Sección 8 — Finetune en Isaac Lab

**Precondición:** al menos 3 PKLs aprobados (§6.4 + §7 cerrado por cada uno) listos para entrenamiento. Display NO requerido — corre headless.

El finetune no se lanza con un PKL individual. Se lanza con una carpeta que contiene todas las variantes/actores del skill. Esta sección organiza esa carpeta, verifica el set completo, y lanza el entrenamiento. El criterio de cierre es convergencia en wandb — no que el entrenamiento termine.

## 8.0 — CONFIG de entrenamiento

DIR_TRAINING es la carpeta con todos los PKLs aprobados para este run. No mezclar skills distintos en la misma carpeta.

In [ ]:
DIR_TRAINING="/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2" # <- carpeta con PKLs aprobados del skill
CHECKPOINT="/home/udirobotika/blea/GR00T-WholeBodyControl/sonic_release/last.pt"
NUM_ENVS=128
NUM_ITERS=1000
WANDB_ENTITY="crisguycabs-udi"        # <- tu usuario de wandb
WANDB_PROJECT="TRL_G1_Track"          # <- no cambiar: proyecto fijo
WANDB_RUN_NAME="macarena"   # <- editar por run: skill_actor_version

echo "Carpeta de entrenamiento : $DIR_TRAINING"
echo "Checkpoint base          : $CHECKPOINT"
echo "Entornos paralelos       : $NUM_ENVS"
echo "Iteraciones              : $NUM_ITERS"
echo "wandb project            : $WANDB_PROJECT"
echo "wandb run name           : $WANDB_RUN_NAME"
echo ""
echo "PKLs en la carpeta:"
ls -lh $DIR_TRAINING/*.pkl
echo ""
echo "Total: $(ls $DIR_TRAINING/*.pkl | wc -l) PKLs"

✅ Listo cuando: lista los PKLs correctos, sin end-to-end-test ni archivos de prueba.

## 8.1 — Verificar el set completo:

In [ ]:
python - << 'EOF'
import joblib, numpy as np, pathlib

DIR_TRAINING="/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2" # <- carpeta con PKLs aprobados del skill
FPS_ESPERADO = 50

pkls = sorted(pathlib.Path(DIR_TRAINING).glob("*.pkl"))
print(f"Carpeta : {DIR_TRAINING}")
print(f"PKLs    : {len(pkls)}\n")

ok_count = 0
for pkl in pkls:
    try:
        d   = joblib.load(pkl)
        key = list(d.keys())[0]
        seq = d[key]
        dof = np.asarray(seq["dof"])
        rr  = np.asarray(seq["root_rot"])
        norms = np.linalg.norm(rr, axis=1)

        shape_ok = dof.shape[1] == 29
        quat_ok  = norms.min() > 0.999 and norms.max() < 1.001
        fps_ok   = seq.get("fps") == FPS_ESPERADO

        status = "✅" if (shape_ok and quat_ok and fps_ok) else "⚠️"
        if shape_ok and quat_ok and fps_ok:
            ok_count += 1
        print(f"{status} {pkl.name}")
        print(f"   key={key}  T={dof.shape[0]}  ({dof.shape[0]/FPS_ESPERADO:.1f}s @ {seq.get('fps')}fps)")
        print(f"   dof={dof.shape}  root_rot norm OK={quat_ok}")
        if not (shape_ok and quat_ok and fps_ok):
            print(f"   ⚠️ PROBLEMA — no incluir en entrenamiento")
    except Exception as e:
        print(f"⚠️ {pkl.name}  ERROR: {e}")

print(f"\n{'='*50}")
print(f"Resultado: {ok_count}/{len(pkls)} PKLs listos")
if ok_count == len(pkls) and len(pkls) > 0:
    print("→ Set completo. Proceder a §8.2.")
else:
    print("→ Revisar PKLs con ⚠️ antes de entrenar.")
EOF

✅ Listo cuando: todos los PKLs muestran ✅ y el mensaje dice "Proceder a §8.2".

## 8.2 — Configurar wandb:

In [ ]:
wandb status

api_key distinto de null y cuenta correcta → listo, ir a §8.3
api_key null → autenticar:

In [ ]:
wandb login

Pega el API key de https://wandb.ai/authorize cuando lo pida. Queda guardado permanentemente — no hay que repetirlo en sesiones futuras.

api_key distinto de null pero cuenta equivocada → cerrar sesión primero:


In [ ]:
wandb login --relogin

En ambos casos pegar el API key de https://wandb.ai/authorize.

Verificar que quedó bien:

In [ ]:
python3 -c "import wandb; wandb.init(project='test-conexion'); print('Conexion exitosa'); wandb.finish()"

## 8.3 — Lanzar finetune:

In [ ]:
watch -n 1 nvidia-smi

In [ ]:
cd ~/blea/GR00T-WholeBodyControl
~/IsaacLab/isaaclab.sh -p gear_sonic/train_agent_trl.py \
  +exp=manager/universal_token/all_modes/sonic_release \
  +checkpoint=$CHECKPOINT \
  num_envs=$NUM_ENVS \
  headless=True \
  ++algo.config.num_learning_iterations=$NUM_ITERS \
  ++algo.config.load_optimizer=False \
  ++use_wandb=true \
  ++wandb.project=$WANDB_PROJECT \
  ++wandb.name=$WANDB_RUN_NAME \
  manager_env.commands.motion.motion_lib_cfg.motion_file=$DIR_TRAINING

Si no sirve el primero, corra el siguiente, si le sirvio el primero GG

In [ ]:
cd ~/blea/GR00T-WholeBodyControl
~/IsaacLab/isaaclab.sh -p gear_sonic/train_agent_trl.py \
  +exp=manager/universal_token/all_modes/sonic_release \
  +checkpoint=$CHECKPOINT \
  num_envs=$NUM_ENVS \
  headless=True \
  ++algo.config.num_learning_iterations=$NUM_ITERS \
  ++algo.config.load_optimizer=False \
  ++use_wandb=true \
  ++wandb.project=$WANDB_PROJECT \
  ++wandb.name=$WANDB_RUN_NAME \
  manager_env.commands.motion.motion_lib_cfg.motion_file=$DIR_TRAINING \
  ++manager_env.commands.motion.motion_lib_cfg.adaptive_sampling.adp_samp_failure_rate_max_over_mean=30 \
  ++manager_env.commands.motion.motion_lib_cfg.adaptive_sampling.uniform_sampling_rate=0.15 \
  ++manager_env.terminations.foot_pos_xyz.params.threshold=0.7

In [ ]:
cd ~/blea/GR00T-WholeBodyControl
~/IsaacLab/isaaclab.sh -p gear_sonic/train_agent_trl.py \
  +exp=manager/universal_token/all_modes/sonic_release \
  +checkpoint=$CHECKPOINT \
  num_envs=$NUM_ENVS \
  headless=True \
  ++algo.config.num_learning_iterations=$NUM_ITERS \
  ++algo.config.load_optimizer=False \
  ++use_wandb=true \
  ++wandb.project=$WANDB_PROJECT \
  ++wandb.name=$WANDB_RUN_NAME \
  manager_env.commands.motion.motion_lib_cfg.motion_file=$DIR_TRAINING \
  ++manager_env.commands.motion.motion_lib_cfg.adaptive_sampling.adp_samp_failure_rate_max_over_mean=30 \
  ++manager_env.commands.motion.motion_lib_cfg.adaptive_sampling.uniform_sampling_rate=0.15 \
  ++manager_env.terminations.foot_pos_xyz.params.threshold=0.7 \
  ++manager_env.terminations.ee_body_pos.params.threshold=0.55

⚠️ Variables $DIR_TRAINING, $CHECKPOINT, $NUM_ENVS, $NUM_ITERS, $WANDB_PROJECT y $WANDB_RUN_NAME deben estar definidas — misma terminal que §8.0, o volver a correr ese bloque.

⚠️ Si aparece OOM → bajar NUM_ENVS=64 en §8.0 y relanzar.

⚠️ No cerrar esta terminal — el entrenamiento corre en foreground.

✅ Listo cuando: aparece URL del run en wandb en consola y llegan iteraciones numeradas sin error de shape.

## 8.4 — Gate de convergencia en wandb

Precondición: entrenamiento corriendo en §8.3, AL MENOS las primeras 100 iteraciones completadas.

Esta sección no espera que el entrenamiento termine. El criterio es que failure_rate_mean muestre tendencia descendente en las primeras 100 iteraciones — eso confirma que la política está aprendiendo el skill capturado.

Abrir el dashboard de wandb:

In [ ]:
https://wandb.ai/crisguycabs-udi/TRL_G1_Track

Buscar el run `$WANDB_RUN_NAME` y revisar las siguientes métricas:

**Métrica principal — `failure_rate_mean`:**
- Iteraciones 0–20: valor alto esperado (0.6–0.9), política aún sin convergir
- Iteraciones 20–100: debe mostrar tendencia descendente clara
- Por debajo de 0.3 antes de iteración 100 → convergencia saludable
- Se mantiene por encima de 0.5 después de iteración 100 → problema, detener y revisar

**Métricas de soporte:**
- `episode_length_mean` — debe aumentar con el tiempo
- `policy_loss` — debe decrecer
- `value_loss` — puede oscilar, no es bloqueante

**Checklist de aprobación:**
- [ ] `failure_rate_mean` con tendencia descendente antes de iteración 100
- [ ] Sin spikes de NaN en ninguna métrica
- [ ] GPU memory estable (no creciendo iteración a iteración)

⚠️ Si `failure_rate_mean` no baja de 0.5 en 100 iteraciones → detener con Ctrl+C, revisar que `DIR_TRAINING` apunta a la carpeta correcta y que los PKLs pasaron §8.1.

✅ Gate cerrado cuando: `failure_rate_mean` muestra tendencia descendente sostenida. El entrenamiento puede seguir corriendo hasta `NUM_ITERS`.

La salida del entrenamiento debe ser similar a:

## 8.5 — Reporte final del entrenamiento

**Precondición:** entrenamiento completado (NUM_ITERS iteraciones). Esta sección descarga las métricas desde wandb y genera un reporte de la sesión.

Antes, se debe verificar el ID del run en wandb

In [ ]:
deactivate 2>/dev/null; conda activate isaaclab
cd ~/blea/GR00T-WholeBodyControl
python - << 'EOF'
import wandb

WANDB_ENTITY  = "crisguycabs-udi"   # <- copiar de §8.0
WANDB_PROJECT = "TRL_G1_Track"      # <- copiar de §8.0

api  = wandb.Api()
runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}", order="-created_at")

print(f"{'#':<3} {'ID':<12} {'Estado':<10} {'Creado':<22} {'Nombre'}")
print("-" * 90)
for i, r in enumerate(list(runs)[:10]):
    created = r.created_at[:19].replace("T", " ") if r.created_at else "N/A"
    print(f"{i+1:<3} {r.id:<12} {r.state:<10} {created:<22} {r.name}")
EOF

anotar el ID del último run, o el run de interes

Saved model checkpoint to logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-20260812_163047/last.pt


In [ ]:
deactivate 2>/dev/null; conda activate isaaclab
cd ~/blea/GR00T-WholeBodyControl
python - << 'EOF'
import wandb
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pathlib, datetime

WANDB_ENTITY   = "crisguycabs-udi"       # <- copiar de §8.0
WANDB_PROJECT  = "TRL_G1_Track"          # <- copiar de §8.0
WANDB_RUN_ID   = "9q7w1178"             # <- copiar el ID del run de §8.4b
DIR_TRAINING   = "/home/udirobotika/blea/GR00T-WholeBodyControl/robot_filtered_motionlib_macarena2"  # <- copiar de §8.0
OUT_DIR        = pathlib.Path(f"logs_rl/reportes/{WANDB_RUN_ID}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

api = wandb.Api()
run = api.run(f"{WANDB_ENTITY}/{WANDB_PROJECT}/{WANDB_RUN_ID}")
print(f"Run encontrado: {run.name}  estado={run.state}")

df = run.history(samples=10000)
csv_path = OUT_DIR / "metrics.csv"
df.to_csv(csv_path, index=False)
print(f"✅ Métricas descargadas: {len(df)} filas → {csv_path}")

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
metricas = [
    ("failure_rate_mean",   "red",   "Failure Rate Mean"),
    ("episode_length_mean", "blue",  "Episode Length Mean"),
    ("policy_loss",         "green", "Policy Loss"),
]
for ax, (col, color, label) in zip(axes, metricas):
    if col in df.columns:
        ax.plot(df[col].dropna().values, color=color, linewidth=1.2)
        ax.set_ylabel(label)
        ax.grid(axis='x', linestyle=':', alpha=0.5)
    else:
        ax.set_ylabel(f"{label} (no disponible)")

axes[-1].set_xlabel("Iteración")
fig.suptitle(f"Reporte de entrenamiento — {run.name}", fontsize=11)
plt.tight_layout()
plot_path = OUT_DIR / "training_report.png"
plt.savefig(plot_path, dpi=120)
print(f"✅ Gráfico guardado → {plot_path}")

total_iters   = len(df)
final_failure = df["failure_rate_mean"].dropna().iloc[-1] if "failure_rate_mean" in df.columns and not df["failure_rate_mean"].dropna().empty else "N/A"
min_failure   = df["failure_rate_mean"].dropna().min()    if "failure_rate_mean" in df.columns and not df["failure_rate_mean"].dropna().empty else "N/A"
final_str     = f"{final_failure:.4f}" if isinstance(final_failure, float) else str(final_failure)
min_str       = f"{min_failure:.4f}"   if isinstance(min_failure, float)   else str(min_failure)

resumen = f"""
REPORTE DE ENTRENAMIENTO
========================
Run ID        : {WANDB_RUN_ID}
Run name      : {run.name}
Proyecto      : {WANDB_ENTITY}/{WANDB_PROJECT}
Carpeta PKLs  : {DIR_TRAINING}
Fecha         : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}
Iteraciones   : {total_iters}

failure_rate_mean final : {final_str}
failure_rate_mean min   : {min_str}

Veredicto: {'✅ Convergió' if isinstance(final_failure, float) and final_failure < 0.3 else '⚠️ Revisar — failure rate alto al final'}

Archivos generados:
  {csv_path}
  {plot_path}
"""
print(resumen)
report_path = OUT_DIR / "resumen.txt"
report_path.write_text(resumen)
print(f"✅ Resumen guardado → {report_path}")
EOF

✅ Listo cuando: genera metrics.csv, training_report.png y resumen.txt en logs_rl/reportes/<WANDB_RUN_NAME>/.

⚠️ WANDB_RUN_NAME debe coincidir exactamente con el nombre del run en wandb — copiar de §8.0.

---

# Sección 9 — Preparación del modelo para deploy físico

**Precondición:** §8 completado — checkpoint `last.pt` disponible, §7 cerrado con gate visual ✅.

Esta sección cubre la cadena: checkpoint → ONNX export → verificación de shapes → eval IsaacSim (sim2sim) → prep TRT en Jetson.

⚠️ **No pasar al robot físico sin cerrar §9.3 (IsaacSim eval) con inspección visual.**

## 9.1 — CONFIG

In [ ]:
# §9.0 CONFIG — copiar de §8.0 y §8.5
CHECKPOINT="logs_rl/TRL_G1_Track/manager/universal_token/all_modes/sonic_release_test-<TIMESTAMP>/last.pt"
# ^ Editar con el path exacto del §8.5 reporte final

MOTION_FILE="${BASE_BLEA}/robot_filtered_motionlib_${MOTION_KEY}/${OUTPUT_NAME}.pkl"
# ^ Motionlib PKL para IsaacSim eval (NO el vis PKL — son incompatibles)

DEPLOY_DIR="${BASE_BLEA}/gear_sonic_deploy/policy/release"

echo "Checkpoint : ${CHECKPOINT}"
echo "Motion file: ${MOTION_FILE}"
echo "Deploy dir : ${DEPLOY_DIR}"
ls -lh "${DEPLOY_DIR}/"

## 9.2 — Export ONNX

In [ ]:
# §9.2 — Export ONNX desde checkpoint
# Verificar primero qué scripts de export existen
ls ~/blea/GR00T-WholeBodyControl/gear_sonic/export*.py 2>/dev/null || \
  ls ~/blea/GR00T-WholeBodyControl/gear_sonic/*export*.py 2>/dev/null || \
  echo "⚠️ Script de export no encontrado — buscar en gear_sonic/"

# Lanzar export (ajustar nombre del script según ls anterior)
cd ~/blea/GR00T-WholeBodyControl
~/IsaacLab/isaaclab.sh -p gear_sonic/export_model.py \
  +checkpoint=$CHECKPOINT

# Verificar archivos generados (buscar en directorio del checkpoint y en raíz)
CHECKPOINT_DIR=$(dirname $CHECKPOINT)
echo ""
echo "ONNX generados:"
find $CHECKPOINT_DIR -name "*.onnx" -o -name "model_*.onnx" 2>/dev/null | head -20

## 9.3 — Mover ONNXs a deploy y verificar shapes

In [ ]:
# §9.3 — Instalar en carpeta de deploy y verificar shapes
# Ajustar los nombres exactos según la salida del export
ENCODER_ONNX=$(find $(dirname $CHECKPOINT) -name "*encoder*.onnx" | head -1)
DECODER_ONNX=$(find $(dirname $CHECKPOINT) -name "*decoder*.onnx" | head -1)

echo "Encoder encontrado: $ENCODER_ONNX"
echo "Decoder encontrado: $DECODER_ONNX"

# Backup de los ONNXs actuales
cp "${DEPLOY_DIR}/model_encoder.onnx" "${DEPLOY_DIR}/model_encoder.onnx.bak" 2>/dev/null && echo "Backup encoder OK"
cp "${DEPLOY_DIR}/model_decoder.onnx" "${DEPLOY_DIR}/model_decoder.onnx.bak" 2>/dev/null && echo "Backup decoder OK"

# Instalar los nuevos
cp "$ENCODER_ONNX" "${DEPLOY_DIR}/model_encoder.onnx"
cp "$DECODER_ONNX" "${DEPLOY_DIR}/model_decoder.onnx"

echo ""
echo "Estado final de ${DEPLOY_DIR}/:"
ls -lh "${DEPLOY_DIR}/"*.onnx

In [ ]:
# Verificación de shapes — requiere conda isaaclab (tiene onnxruntime)
conda activate isaaclab

/home/udirobotika/anaconda3/envs/isaaclab/bin/python3 - << 'EOF'
import onnxruntime as ort

DEPLOY_DIR = "/home/udirobotika/blea/GR00T-WholeBodyControl/gear_sonic_deploy/policy/release"

for name, path in [("ENCODER", f"{DEPLOY_DIR}/model_encoder.onnx"),
                   ("DECODER", f"{DEPLOY_DIR}/model_decoder.onnx")]:
    try:
        sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
        print(f"=== {name} ({path.split('/')[-1]}) ===")
        for inp in sess.get_inputs():
            print(f"  IN  {inp.name:<30} shape={str(inp.shape):<20} dtype={inp.type}")
        for out in sess.get_outputs():
            print(f"  OUT {out.name:<30} shape={str(out.shape):<20} dtype={out.type}")
        print()
    except Exception as e:
        print(f"⛔ {name}: {e}")

print("Shapes esperadas:")
print("  Encoder: input [1, 1751] → output [1, 64, token_dim]")
print("  Decoder: input [1, 994]  → output [1, 29]  (29 joints)")
EOF

## 9.4 — Eval IsaacSim (sim2sim gate)

**Precondición:** ONNXs instalados en `policy/release/`, shapes verificadas en §9.3.

El eval carga el checkpoint directamente (no los ONNXs) y reproduce el movimiento de referencia con la política finetuned. El robot debe ejecutar el movimiento capturado de forma reconocible.

⚠️ Usar el motionlib PKL (`robot_filtered_motionlib_*/`), NO el vis PKL. El eval de IsaacLab usa un formato diferente al deploy.

✅ Gate cerrado cuando: el robot virtual ejecuta el movimiento reconocible durante toda la secuencia. El renderer puede ser amarillo (Storm fallback en Blackwell) — eso no es error.

In [ ]:
# §9.4 — IsaacSim eval
# Verificar variables definidas en §9.1
echo "Checkpoint : $CHECKPOINT"
echo "Motion file: $MOTION_FILE"
echo ""

cd ~/blea/GR00T-WholeBodyControl
~/IsaacLab/isaaclab.sh -p gear_sonic/eval_agent_trl.py \
  +checkpoint=$CHECKPOINT \
  ++num_envs=1 \
  ++headless=False \
  ++run_once=True \
  ++use_wandb=false \
  ++manager_env.commands.motion.motion_lib_cfg.motion_file="$MOTION_FILE"

## 9.5 — Compilación TRT en Jetson ⏸️ DIFERIDO

**Precondición:** §9.4 cerrado, robot G1 disponible con Jetson accesible por SSH.

Los TRT engines son device-specific — los compilados para el Alienware (x86_64, RTX 5060 Ti) no corren en el Jetson (aarch64). Cada nuevo par de ONNXs requiere recompilación en el Jetson.

**Pasos cuando el robot esté disponible:**

1. Transferir ONNXs al Jetson:
```bash
scp gear_sonic_deploy/policy/release/model_encoder.onnx \
    gear_sonic_deploy/policy/release/model_decoder.onnx \
    unitree@192.168.123.164:~/gear_sonic_deploy/policy/release/
```

2. En el Jetson — borrar TRT caches anteriores:
```bash
ssh unitree@192.168.123.164
rm -f ~/gear_sonic_deploy/policy/release/*.trt
```

3. El binary recompila automáticamente al primer arranque (cache miss → TRT build):
```bash
# Desde el Jetson, dentro de tmux:
cd ~/gear_sonic_deploy
./target/release/g1_deploy_onnx_ref eth0 \
  policy/release/model_decoder.onnx \
  reference/${MOTION_NAME}/ \
  --encoder-file policy/release/model_encoder.onnx \
  --planner-file planner/target_vel/V2/planner_sonic_opset14_cf2.onnx \
  --obs-config policy/release/observation_config.yaml \
  --input-type keyboard 2>&1 | tee trt_build.log
```

4. Esperar "Init Done" en el log (~60–90s primera vez por build TRT).

⚠️ Verificar JetPack version antes de recompilar: `cat /etc/nv_tegra_release`. La deuda TRT (JP5 vs JP6) sigue abierta — ver issue en notas de §8.4.

✅ Gate cerrado cuando: "Init Done" aparece en log, dims `994/1751` confirmados, binary en modo keyboard listo.